In [25]:
import os
import json
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from collections import Counter

from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    accuracy_score, f1_score, confusion_matrix,
    classification_report
)
from sklearn.preprocessing import LabelEncoder
import joblib

os.environ['MPLCONFIGDIR'] = f"/tmp/matplotlib_{os.environ.get('USER', 'user')}"

USER = os.environ.get('USER', 'sigursteinsson1')
FINAL_PROJECT_DIR = Path(f"/p/project1/training2600/{USER}/Final_Project")
TRAINING_DATA_DIR = Path(f"/p/project1/training2600/{USER}/training_data_checkpoint")
RESULTS_DIR       = FINAL_PROJECT_DIR / "results"
MODELS_DIR        = FINAL_PROJECT_DIR / "models"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)


In [24]:
# only picking out the relevant classes that have more or equal to 200 samples
PROCESSED_DIR = FINAL_PROJECT_DIR / "processed_data" # get only the processed classes that survived the filtering

surviving_class_names = sorted([
    d.name for d in (PROCESSED_DIR / "train").iterdir() if d.is_dir()
])

CORINE_CLASSES = { # same collection as in dataprep, we will use reverse lookup
    1: "continuous_urban_fabric", 2: "discontinuous_urban_fabric",
    3: "industrial_or_commercial_units", 4: "road_and_rail_networks",
    5: "port_areas", 6: "airports", 7: "mineral_extraction_sites",
    8: "dump_sites", 9: "construction_sites", 10: "green_urban_areas",
    11: "sport_and_leisure_facilities", 12: "non_irrigated_arable_land",
    13: "permanently_irrigated_land", 14: "rice_fields", 15: "vineyards",
    16: "fruit_trees_and_berry_plantations", 17: "olive_groves",
    18: "pastures", 19: "annual_crops_with_permanent_crops",
    20: "complex_cultivation_patterns", 21: "agriculture_with_natural_vegetation",
    22: "agro_forestry_areas", 23: "broad_leaved_forest",
    24: "coniferous_forest", 25: "mixed_forest", 26: "natural_grasslands",
    27: "moors_and_heathland", 28: "sclerophyllous_vegetation",
    29: "transitional_woodland_shrub", 30: "beaches_dunes_sands",
    31: "bare_rocks", 32: "sparsely_vegetated_areas", 33: "burnt_areas",
    34: "glaciers_and_perpetual_snow", 35: "inland_marshes", 36: "peat_bogs",
    37: "salt_marshes", 38: "salines", 39: "intertidal_flats",
    40: "water_courses", 41: "water_bodies", 42: "coastal_lagoons",
    43: "estuaries", 44: "sea_and_ocean",
}

NAME_TO_ID = {v: k for k, v in CORINE_CLASSES.items()}

# surviving classes:
CLASS_NAMES = {
    NAME_TO_ID[name]: name
    for name in surviving_class_names
    if name in NAME_TO_ID
}

for s in CLASS_NAMES: print(s, "Survived")

23 Survived
24 Survived
25 Survived
27 Survived
36 Survived
29 Survived
41 Survived
40 Survived


In [23]:
# since random forest and MLP dont understand spatial structure we want them to match the future prithvi model structure, 
# so the arrays we define here in the baseline model we just get a flat vector of numbers per sample, which add up to 144
# prithvi will however be able to learn from the 3x3x16 tensor shape

def load_split(split):
    patches = np.load(TRAINING_DATA_DIR / f"patches_{split}.npz")["patches"]
    labels  = np.load(TRAINING_DATA_DIR / f"labels_{split}.npy")
    # truncate only to the surviving classes
    kept_classes = set(CLASS_NAMES.keys())
    mask = np.array([int(l) in kept_classes for l in labels])
    patches, labels = patches[mask], labels[mask]
    # Flatten (3,3,16) -> 144 features for sklearn
    X = patches.reshape(len(patches), -1).astype(np.float32) # float 64 replacement, since the arrays arent that large they fit in float32 instead of flaot64
    y = labels.astype(np.int32)
    return X, y

X_train, y_train = load_split("train")
X_val,   y_val   = load_split("val")
X_test,  y_test  = load_split("test")

# Encode labels to 0-indexed for sklearn since the class numbers are arbitrary, not continuous
le = LabelEncoder()
le.fit(sorted(CLASS_NAMES.keys())) 
y_train_enc = le.transform(y_train)
y_val_enc   = le.transform(y_val)
y_test_enc  = le.transform(y_test)

class_names_ordered = [CLASS_NAMES[c] for c in le.classes_]

print(f"Train: {X_train.shape} \nVal: {X_val.shape} \nTest: {X_test.shape}")
print(f"Classes: {list(le.classes_)}")
print(f"Feature vector size: {X_train.shape[1]} (3×3×16 flattened)")

Train: (19838, 144) 
Val: (9933, 144) 
Test: (9929, 144)
Classes: [23, 24, 25, 27, 29, 36, 40, 41]
Feature vector size: 144 (3×3×16 flattened)


In [26]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report
import joblib

print("Training Random Forest (naive)")
print(f"  Input shape: {X_train.shape}")
print(f"  Classes: {list(le.classes_)}\n")

# Config
rf = RandomForestClassifier(
    n_estimators=500,
    max_depth=None,
    min_samples_leaf=2,
    n_jobs=-1,
    random_state=42,
    class_weight="balanced",
)
rf.fit(X_train, y_train_enc)
print("Training complete.")

rf_val_pred  = rf.predict(X_val)
rf_test_pred = rf.predict(X_test)

rf_val_oa  = accuracy_score(y_val_enc,  rf_val_pred)
rf_test_oa = accuracy_score(y_test_enc, rf_test_pred)
rf_val_f1  = f1_score(y_val_enc,  rf_val_pred,  average="macro", zero_division=0)
rf_test_f1 = f1_score(y_test_enc, rf_test_pred, average="macro", zero_division=0)

print(f"Random Forest (naive):")
print(f"  Val  — OA: {rf_val_oa:.4f}   Macro F1: {rf_val_f1:.4f}")
print(f"  Test — OA: {rf_test_oa:.4f}   Macro F1: {rf_test_f1:.4f}")
print(classification_report(y_test_enc, rf_test_pred,
      target_names=class_names_ordered, digits=3, zero_division=0))

joblib.dump(rf, MODELS_DIR / "random_forest_naive.pkl")
print("Model saved.")

Training Random Forest (naive)
  Input shape: (19838, 144)
  Classes: [23, 24, 25, 27, 29, 36, 40, 41]

Training complete.
Random Forest (naive):
  Val  — OA: 0.4125   Macro F1: 0.1199
  Test — OA: 0.4905   Macro F1: 0.1362
                             precision    recall  f1-score   support

        broad_leaved_forest      0.205     0.131     0.160       314
          coniferous_forest      0.539     0.851     0.660      5028
               mixed_forest      1.000     0.002     0.005       873
        moors_and_heathland      0.111     0.004     0.008       251
transitional_woodland_shrub      0.000     0.000     0.000       700
                  peat_bogs      0.307     0.222     0.258      2458
              water_courses      0.000     0.000     0.000        46
               water_bodies      0.000     0.000     0.000       259

                   accuracy                          0.490      9929
                  macro avg      0.270     0.151     0.136      9929
               

In [27]:
from sklearn.neural_network import MLPClassifier

print("Training MLP (naive)")

mlp = MLPClassifier(
    hidden_layer_sizes=(256, 128, 64),
    activation="relu",
    max_iter=200,
    early_stopping=True,
    n_iter_no_change=10,
    validation_fraction=0.1,
    random_state=42,
    verbose=True,
)
mlp.fit(X_train, y_train_enc)

mlp_val_pred  = mlp.predict(X_val)
mlp_test_pred = mlp.predict(X_test)

mlp_val_oa  = accuracy_score(y_val_enc,  mlp_val_pred)
mlp_test_oa = accuracy_score(y_test_enc, mlp_test_pred)
mlp_val_f1  = f1_score(y_val_enc,  mlp_val_pred,  average="macro", zero_division=0)
mlp_test_f1 = f1_score(y_test_enc, mlp_test_pred, average="macro", zero_division=0)

print(f"\nMLP (naive):")
print(f"  Val  — OA: {mlp_val_oa:.4f}   Macro F1: {mlp_val_f1:.4f}")
print(f"  Test — OA: {mlp_test_oa:.4f}   Macro F1: {mlp_test_f1:.4f}")
print(classification_report(y_test_enc, mlp_test_pred,
      target_names=class_names_ordered, digits=3, zero_division=0))

joblib.dump(mlp, MODELS_DIR / "mlp_naive.pkl")
print("Model saved.")

Training MLP (naive)
Iteration 1, loss = 1.57464571
Validation score: 0.454133
Iteration 2, loss = 1.51272670
Validation score: 0.450101
Iteration 3, loss = 1.50194965
Validation score: 0.448085
Iteration 4, loss = 1.49211662
Validation score: 0.462198
Iteration 5, loss = 1.47867489
Validation score: 0.458669
Iteration 6, loss = 1.47119897
Validation score: 0.455645
Iteration 7, loss = 1.46530601
Validation score: 0.458669
Iteration 8, loss = 1.45870378
Validation score: 0.467238
Iteration 9, loss = 1.46005810
Validation score: 0.460685
Iteration 10, loss = 1.44995892
Validation score: 0.467238
Iteration 11, loss = 1.44657096
Validation score: 0.460181
Iteration 12, loss = 1.44266332
Validation score: 0.461190
Iteration 13, loss = 1.44099856
Validation score: 0.454637
Iteration 14, loss = 1.43361431
Validation score: 0.467238
Iteration 15, loss = 1.43107406
Validation score: 0.467742
Iteration 16, loss = 1.42798158
Validation score: 0.469254
Iteration 17, loss = 1.42305236
Validation s

In [16]:
# Balanced training set via undersampling
MAX_PER_CLASS = 1500
rng = np.random.default_rng(42)

balanced_idx = []
for cls_idx in np.unique(y_train_enc):
    idx = np.where(y_train_enc == cls_idx)[0]
    if len(idx) > MAX_PER_CLASS:
        idx = rng.choice(idx, size=MAX_PER_CLASS, replace=False)
    balanced_idx.extend(idx.tolist())

X_train_bal = X_train[balanced_idx]
y_train_bal = y_train_enc[balanced_idx]

print(f"Balanced training set: {len(X_train_bal):,} samples")
from collections import Counter
for cls_idx, count in sorted(Counter(y_train_bal.tolist()).items()):
    print(f"  {class_names_ordered[cls_idx]:<35} {count:>5,}")

Balanced training set: 8,473 samples
  broad_leaved_forest                 1,500
  coniferous_forest                   1,500
  mixed_forest                        1,220
  moors_and_heathland                   713
  transitional_woodland_shrub         1,362
  peat_bogs                           1,500
  water_courses                         136
  water_bodies                          542


In [17]:
# Improved RF
print("Training improved Random Forest (undersampled)")
rf_bal = RandomForestClassifier(
    n_estimators=500,
    min_samples_leaf=2,
    n_jobs=-1,
    random_state=42,
)
rf_bal.fit(X_train_bal, y_train_bal)

rf_bal_test_pred = rf_bal.predict(X_test)
rf_bal_val_pred  = rf_bal.predict(X_val)
rf_bal_test_oa   = accuracy_score(y_test_enc, rf_bal_test_pred)
rf_bal_test_f1   = f1_score(y_test_enc, rf_bal_test_pred, average="macro", zero_division=0)
rf_bal_val_oa    = accuracy_score(y_val_enc,  rf_bal_val_pred)
rf_bal_val_f1    = f1_score(y_val_enc,  rf_bal_val_pred,  average="macro", zero_division=0)

print(f"  Val  — OA: {rf_bal_val_oa:.4f}   Macro F1: {rf_bal_val_f1:.4f}")
print(f"  Test — OA: {rf_bal_test_oa:.4f}   Macro F1: {rf_bal_test_f1:.4f}")
print(classification_report(y_test_enc, rf_bal_test_pred,
      target_names=class_names_ordered, digits=3, zero_division=0))
joblib.dump(rf_bal, MODELS_DIR / "random_forest_balanced.pkl")


Training improved Random Forest (undersampled)...
  Val  — OA: 0.2279   Macro F1: 0.1256
  Test — OA: 0.2383   Macro F1: 0.1225
                             precision    recall  f1-score   support

        broad_leaved_forest      0.072     0.570     0.128       314
          coniferous_forest      0.558     0.286     0.378      5028
               mixed_forest      0.086     0.085     0.085       873
        moors_and_heathland      0.034     0.016     0.022       251
transitional_woodland_shrub      0.086     0.250     0.128       700
                  peat_bogs      0.273     0.202     0.232      2458
              water_courses      0.000     0.000     0.000        46
               water_bodies      0.028     0.004     0.007       259

                   accuracy                          0.238      9929
                  macro avg      0.142     0.177     0.122      9929
               weighted avg      0.367     0.238     0.270      9929



['/p/project1/training2600/sigursteinsson1/Final_Project/models/random_forest_balanced.pkl']

In [18]:
# Improved MLP
print("\nTraining improved MLP (undersampled)...")

# Config
mlp_bal = MLPClassifier(
    hidden_layer_sizes=(256, 128, 64),
    activation="relu",
    max_iter=200,
    early_stopping=True,
    n_iter_no_change=10,
    validation_fraction=0.1,
    random_state=42,
    verbose=False,
)
mlp_bal.fit(X_train_bal, y_train_bal)

mlp_bal_test_pred = mlp_bal.predict(X_test)
mlp_bal_val_pred  = mlp_bal.predict(X_val)
mlp_bal_test_oa   = accuracy_score(y_test_enc, mlp_bal_test_pred)
mlp_bal_test_f1   = f1_score(y_test_enc, mlp_bal_test_pred, average="macro", zero_division=0)
mlp_bal_val_oa    = accuracy_score(y_val_enc,  mlp_bal_val_pred)
mlp_bal_val_f1    = f1_score(y_val_enc,  mlp_bal_val_pred,  average="macro", zero_division=0)

print(f"  Val  OA: {mlp_bal_val_oa:.4f}   Macro F1: {mlp_bal_val_f1:.4f}")
print(f"  Test OA: {mlp_bal_test_oa:.4f}   Macro F1: {mlp_bal_test_f1:.4f}")
print(classification_report(y_test_enc, mlp_bal_test_pred,
      target_names=class_names_ordered, digits=3, zero_division=0))
joblib.dump(mlp_bal, MODELS_DIR / "mlp_balanced.pkl")


Training improved MLP (undersampled)...
  Val  — OA: 0.2122   Macro F1: 0.1245
  Test — OA: 0.2100   Macro F1: 0.1216
                             precision    recall  f1-score   support

        broad_leaved_forest      0.070     0.519     0.123       314
          coniferous_forest      0.549     0.201     0.295      5028
               mixed_forest      0.135     0.139     0.137       873
        moors_and_heathland      0.032     0.064     0.043       251
transitional_woodland_shrub      0.087     0.239     0.128       700
                  peat_bogs      0.250     0.246     0.248      2458
              water_courses      0.000     0.000     0.000        46
               water_bodies      0.000     0.000     0.000       259

                   accuracy                          0.210      9929
                  macro avg      0.140     0.176     0.122      9929
               weighted avg      0.361     0.210     0.237      9929



['/p/project1/training2600/sigursteinsson1/Final_Project/models/mlp_balanced.pkl']

In [21]:
# Final comparison table between RF and MLP
results = {
    "RF (naive)":       {"test_oa": rf_test_oa,      "test_macro_f1": rf_test_f1},
    "MLP (naive)":      {"test_oa": mlp_test_oa,     "test_macro_f1": mlp_test_f1},
    "RF (balanced)":    {"test_oa": rf_bal_test_oa,  "test_macro_f1": rf_bal_test_f1},
    "MLP (balanced)":   {"test_oa": mlp_bal_test_oa, "test_macro_f1": mlp_bal_test_f1},
    "Prithvi":          {"test_oa": None,             "test_macro_f1": None},
}

print(f"\n{'Model':<20} {'Test OA':>10} {'Test Macro F1':>14}")
print("----------------------------------------------")
for name, r in results.items():
    oa  = f"{r['test_oa']:.4f}"  if r['test_oa']  is not None else "pending"
    f1  = f"{r['test_macro_f1']:.4f}" if r['test_macro_f1'] is not None else "pending"
    print(f"{name:<20} {oa:>10} {f1:>14}")

import json
with open(RESULTS_DIR / "all_results.json", "w") as f:
    json.dump({
        "rf_naive":       {"test_oa": rf_test_oa,      "test_macro_f1": rf_test_f1},
        "mlp_naive":      {"test_oa": mlp_test_oa,     "test_macro_f1": mlp_test_f1},
        "rf_balanced":    {"test_oa": rf_bal_test_oa,  "test_macro_f1": rf_bal_test_f1},
        "mlp_balanced":   {"test_oa": mlp_bal_test_oa, "test_macro_f1": mlp_bal_test_f1},
    }, f, indent=2)
print("Saved all results to all_results.json")


Model                   Test OA  Test Macro F1
----------------------------------------------
RF (naive)               0.4905         0.1362
MLP (naive)              0.4803         0.1256
RF (balanced)            0.2383         0.1225
MLP (balanced)           0.2100         0.1216
Prithvi                 pending        pending
Saved all results to all_results.json
